## Veranschaulichung der Kameraprojektion mit homogenen Koordinaten

Dieses Jupyter Noteboom veranschaulicht die Projektion von Punkten aus einem 3D-Koordinatensystem in das Koordinatensystem einer Kamera mithilfe von homogenen Koordinaten.

- Es werden die Koordinaten von verschiedenen 3D Objekten (Kugeln und eine Ebene) importiert
- Die Koordinaten werden in homogene Koordinaten umgewandelt
- Die intrinsischen und extrinsischen Projektionsmatrizen werden definiert 
- Die Projektion der 3D-Punkte in das 2D-Bildkoordinatensystem der Kamera wird durchgeführt
- Die ergebnisse werden zurück in kartesische Koordinaten umgewandelt
- visualisierung der 3D-Objekte und der projizierten Punkte im 2D-Bildkoordinatensystem der Kamera

In diesem Notebook sind keine Programmieraufgaben umzusetzten. Schauen Sie sich das Notebook an, sodass Sie die einzelnen Schritte nachvollziehen können.


#### Bibliotheken inportieren

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#### Definition von Hilfsfunktionen
Um die Projektion vom 3D-Weltkoortdinatensysrem in das 2D-Bildkoordinatensysrem durchführen zu können benötigt man die intrinsischen und die extrinsischen Projektionsmatrizen. Diese werden über Hilfsfunktionen definiert. Die Matrizen wurden hier jeweils in homogener Koordinatendarstellung erstellt, damit die Transformationen als Matrixmultiplikationen durchgeführt werden können. 

In [2]:
# Definition Rotationmatrizen um jeweils eine Achse
def rotation_matrix_z(alpha_degree):
    """Create a 4x4 rotation matrix around the z-axis in homogeneous coordinates.
    
    Args:
        alpha_degree (float): Rotation angle in degrees.
    
    Returns:
        numpy.ndarray: 4x4 rotation matrix around z-axis.
    """
    alpha_radian = np.deg2rad(alpha_degree)

    rotMat =np.array([
    [np.cos(alpha_radian), -np.sin(alpha_radian), 0, 0],
    [np.sin(alpha_radian), np.cos(alpha_radian), 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]])
    return rotMat

################
################

def rotation_matrix_y(alpha_degree):
    """Create a 4x4 rotation matrix around the y-axis in homogeneous coordinates.
    
    Args:
        alpha_degree (float): Rotation angle in degrees.
    
    Returns:
        numpy.ndarray: 4x4 rotation matrix around y-axis.
    """
    alpha_radian = np.deg2rad(alpha_degree)

    rotMat = np.array([
    [np.cos(alpha_radian), 0, np.sin(alpha_radian), 0],
    [0, 1, 0, 0],
    [-np.sin(alpha_radian), 0, np.cos(alpha_radian), 0],
    [0, 0, 0, 1]])
    return rotMat

################
################

def rotation_matrix_x(alpha_degree):
    """Create a 4x4 rotation matrix around the x-axis in homogeneous coordinates.
    
    Args:
        alpha_degree (float): Rotation angle in degrees.
    
    Returns:
        numpy.ndarray: 4x4 rotation matrix around x-axis.
    """
    alpha_radian = np.deg2rad(alpha_degree)

    rotMat = np.array([
        [1, 0, 0, 0],
        [0, np.cos(alpha_radian), -np.sin(alpha_radian), 0],
        [0, np.sin(alpha_radian), np.cos(alpha_radian), 0],
        [0, 0, 0, 1]
    ])
    return rotMat

################
################

# Definition Translationsmatrix
def translation_matrix(t_x, t_y, t_z):
    """Create a 4x4 translation matrix in homogeneous coordinates.
    
    This function generates a translation transformation matrix     
    Args:
        t_x (float): Translation distance along the x-axis.
        t_y (float): Translation distance along the y-axis.
        t_z (float): Translation distance along the z-axis.
    
    Returns:
        numpy.ndarray: 4x4 translation matrix 

    """
    transMat = np.array([
        [1,0,0,t_x],
        [0,1,0,t_y],
        [0,0,1,t_z],
        [0,0,0,1]
        ])
    return transMat

################
################

# Extrinsische Kamerakalibrierung:
def getExtrinsicCameraCalibrationMatrix(rotAngle_x, rotAngle_y, rotAngle_z, t_x, t_y, t_z):
    """Compute the extrinsic camera calibration matrix.
    
    This function calculates the extrinsic camera calibration matrix by combining
    rotation and translation transformations in homogeneous coordinates. 
    Args:
        rotAngle_x (float): Rotation angle around x-axis in Grad.
        rotAngle_y (float): Rotation angle around y-axis in Grad.
        rotAngle_z (float): Rotation angle around z-axis in Grad.
        t_x (float): Translation in x-direction.
        t_y (float): Translation in y-direction.
        t_z (float): Translation in z-direction.
    
    Returns:
        numpy.ndarray: 4x4 extrinsic camera calibration matrix in homogeneous 
            coordinates 
    """
    tranfoMat = translation_matrix(t_x, t_y, t_z) @ rotation_matrix_z(rotAngle_z) @ rotation_matrix_y(rotAngle_y) @ rotation_matrix_x(rotAngle_x)
    return tranfoMat

################
################


def getIntrinsicCameraCalibrationMatrix(focalLength, c_x, c_y):
    """Create the intrinsic camera calibration matrix in homogeneous coordinates
    
    This function generates the intrinsic camera calibration matrix that contains
    the camera's internal parameters including focal length and principal point.
    The matrix is used to project 3D camera coordinates to 2D image coordinates.
    
    Args:
        focalLength (float): Camera focal length in pixels.
        c_x (float): Principal point x-coordinate (image center x-offset).
        c_y (float): Principal point y-coordinate (image center y-offset).
    
    Returns:
        numpy.ndarray: 3x4 intrinsic camera calibration matrix 
    """
    intCameraMatrix = np.array([
        [focalLength, 0, c_x, 0],
        [0, focalLength, c_y, 0],
        [0,0,1,0]
        ])
    return intCameraMatrix


def hom2Cart(coordinates):
    """Convert homogeneous coordinates to Cartesian coordinates.

    Args:
        coordinates (numpy.ndarray): Homogeneous coordinates with shape (n+1, m),
            where the last row contains the scale (homogeneous) component for each column.

    Returns:
        numpy.ndarray: Cartesian coordinates with shape (n, m), obtained by
            dividing each column by its homogeneous scale and removing the last row.
    """
    cart = coordinates / coordinates[-1, :]
    return cart[:-1, :]


def pol2cart(r, theta, phi):
    """Convert spherical (polar) coordinates to Cartesian coordinates.

    Args:
        r (float or array_like): Radius (distance from origin).
        theta (float or array_like): Polar (inclination) angle in radians measured from the positive z-axis.
        phi (float or array_like): Azimuthal angle in radians measured from the positive x-axis in the x-y plane.

    Returns:
        tuple: (x, y, z) Cartesian coordinates. Each element has the same shape as the inputs.
    """
    return  r * np.sin(theta) * np.cos(phi), r *np.sin(theta) * np.sin(phi), r * np.cos(theta)


## Beginn der eigentlichen Berechnung
### Objekte importieren

zur Veranschaulichung der Projektion mit homogenen Koordinaten werden im Raum Koordinatenpunkte von verschiedenen Objekten erstellt. Diese werden jetzt eingeladen.

- Eine Ebene als Bühne
- 3 Kugeln zur veranschaulichung der räumlichen Tife

Diese Punkte befinden sich im Welt Koordinatensysem.

Die Punkte sind als np-array der Form 3xn abgelegt. das heßt `points_Sphere3_world[:,4]` beschreibt beispielsweise die (x,y,z) des 4. Punktes aud der Liste `points_Sphere3_world`

In [3]:
pointsSphere3_world = np.load(r'data/PointsSphere3.npy')
pointsSphere2_world = np.load(r'data/PointsSphere2.npy')
pointsSphere1_world = np.load(r'data/PointsSphere1.npy')
pointsGrid_world = np.load(r'data/PointsGrid.npy')

# combine all points
allPoints_world = np.hstack((pointsSphere3_world, pointsSphere2_world, pointsSphere1_world, pointsGrid_world))

#### Anzeigen der importierten Objekte mit einer 3D-Visualisierung

Um sich zu orientieren werden die importieren Objekte zunächst in einer 3D-Visualisierung dargestellt.

In [4]:
# Create 3D Plotly scatter plot
fig = go.Figure()

# Add each object as a separate trace
fig.add_trace(go.Scatter3d(
    x=pointsGrid_world[0,:], 
    y=pointsGrid_world[1,:], 
    z=pointsGrid_world[2,:],
    mode='markers',
    marker=dict(size=2, color='black'),
    name='Grid'
))

fig.add_trace(go.Scatter3d(
    x=pointsSphere3_world[0,:], 
    y=pointsSphere3_world[1,:], 
    z=pointsSphere3_world[2,:],
    mode='markers',
    marker=dict(size=2, color='red'),
    name='Sphere 3'
))

fig.add_trace(go.Scatter3d(
    x=pointsSphere2_world[0,:], 
    y=pointsSphere2_world[1,:], 
    z=pointsSphere2_world[2,:],
    mode='markers',
    marker=dict(size=2, color='blue'),
    name='Sphere 2'
))

fig.add_trace(go.Scatter3d(
    x=pointsSphere1_world[0,:], 
    y=pointsSphere1_world[1,:], 
    z=pointsSphere1_world[2,:],
    mode='markers',
    marker=dict(size=2, color='green'),
    name='Sphere 1'
))

# Update layout for better 3D visualization
fig.update_layout(
    title='3D World Coordinates',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube'
    ),
    width=800,
    height=600
)

fig.show()

##### Umrechnung in Homogene Koordinaten

zu den Koordinatenlisten wird eine 4. Zeile mit dem Wert 1 hinzugefügt.  Die Punkte bestehen dann aus (x,y,z,1) in homogenen Koordinaten.

In [5]:
# allPoints_world in homogene Koordinaten
allPoints_world_hom = np.vstack((allPoints_world, np.ones((1, allPoints_world.shape[1]))))
pointsSphere3_hom = np.vstack((pointsSphere3_world, np.ones((1, pointsSphere3_world.shape[1]))))
pointsSphere2_hom = np.vstack((pointsSphere2_world, np.ones((1, pointsSphere2_world.shape[1]))))
pointsSphere1_hom = np.vstack((pointsSphere1_world, np.ones((1, pointsSphere1_world.shape[1]))))
pointsGrid_hom = np.vstack((pointsGrid_world, np.ones((1, pointsGrid_world.shape[1]))))

### Projektion
#### Definition der Kameraparameter

Bei der Demonstration in diesem Skrtipt die Transformation der 3D-Weltkoordinaten auf die 2D - Bildebene der Kamera durchgeführt.
Bei der Umrechnung der Koordinaten in Pixelwerte wurde eine inverse Pixelgröße von 200 Pix/mm angenommen.

In [19]:
# intrinsische Kameraparameter

# Brennweite
focalLength = 12

# Pixeldichte
pixeldichte = 300


# Bildmittelpunkt
c_x = 0
c_y = 0


# extrinsische Kameraparameter


# Rotationswinkel um x
angleX = 30
# Rotationswinkel um y
angleY = 45
# Rotationswinkel um z
angleZ = 0

#Translation
t_x = 0
t_y = 0
t_z = 1500



#### Berechnung der Projektionsmatrix
Die Konstruktion der Matrizen wurde oben definiert. Hier werden die Funktionen aufgefrufen um die Matrizen zu erstellen. 

In [20]:
intCamMat = getIntrinsicCameraCalibrationMatrix(focalLength*pixeldichte, 0, 0)
extCamMat = getExtrinsicCameraCalibrationMatrix(angleX,angleY,angleZ, t_x, t_y, t_z)

#### Transformation der Koordinaten und Projektion in das Bildkoordinatensystem der Kamera

Das `@` Zeichen in den Nachfolgenden Zeilen steht für eine Matrixmultiplikation (numpy syntax). 
Die Gleichungen sind von rechts zu lesen. Zunächst werden die Punkte mit der extrinsischen Matrix in das Kamerakoordinatensystem transformiert um dann mit der intrinsischen Matrix auf die Sensorebene / Bildebene der Kamera projiziert zu werden.

**Man Beachte**

Aufgrund der Datenstruktur der Punkte (Punkte sind als 4xn Arrays abgelegt) ist es möglich die gesamte Punkteliste in einem Rechenschritt zu transformieren. Per Definition geht die  Matrixmultiplikation `@` die Spalten (Koordinaten der Punkte) durch und multipliziert diese mit der Transformationsmatrix.

In [21]:
projected_pointsSphere1_hom = intCamMat@extCamMat@pointsSphere1_hom
projected_pointsSphere2_hom = intCamMat@extCamMat@pointsSphere2_hom
projected_pointsSphere3_hom = intCamMat@extCamMat@pointsSphere3_hom
projected_pointsGrid_hom = intCamMat@extCamMat@pointsGrid_hom

#### Umrechnung in karthesische Koordinaten

In [22]:
projected_pointsSphere1 = hom2Cart(projected_pointsSphere1_hom)
projected_pointsSphere2 = hom2Cart(projected_pointsSphere2_hom)
projected_pointsSphere3 = hom2Cart(projected_pointsSphere3_hom)
projected_pointsGrid = hom2Cart(projected_pointsGrid_hom)

### Anzeige der Daten

Die Darstellung erfolgt hioer ausschließlich in einem x-y Plot. D.h. der räumliche Eindruck entsteht durch die Projektion. 



In [23]:
# World coordinates (X-Y view) using Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pointsGrid_world[0,:], 
    y=pointsGrid_world[1,:],
    mode='markers',
    marker=dict(size=3, color='black'),
    name='Grid'
))

fig.add_trace(go.Scatter(
    x=pointsSphere1_world[0,:], 
    y=pointsSphere1_world[1,:],
    mode='markers',
    marker=dict(size=3, color='red'),
    name='Sphere 1'
))

fig.add_trace(go.Scatter(
    x=pointsSphere2_world[0,:], 
    y=pointsSphere2_world[1,:],
    mode='markers',
    marker=dict(size=3, color='blue'),
    name='Sphere 2'
))

fig.add_trace(go.Scatter(
    x=pointsSphere3_world[0,:], 
    y=pointsSphere3_world[1,:],
    mode='markers',
    marker=dict(size=3, color='green'),
    name='Sphere 3'
))

fig.update_layout(
    title='Darstellung der Ausgangspunkte im Weltkoordinatensystem (X-Y Ansicht)',
    xaxis_title='X',
    yaxis_title='Y',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=600,
    height=600
)
fig.show()


# Projected coordinates using Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=projected_pointsGrid[0,:], 
    y=projected_pointsGrid[1,:],
    mode='markers',
    marker=dict(size=3, color='black'),
    name='Grid'
))

fig.add_trace(go.Scatter(
    x=projected_pointsSphere1[0,:], 
    y=projected_pointsSphere1[1,:],
    mode='markers',
    marker=dict(size=3, color='red'),
    name='Sphere 1'
))

fig.add_trace(go.Scatter(
    x=projected_pointsSphere2[0,:], 
    y=projected_pointsSphere2[1,:],
    mode='markers',
    marker=dict(size=3, color='blue'),
    name='Sphere 2'
))

fig.add_trace(go.Scatter(
    x=projected_pointsSphere3[0,:], 
    y=projected_pointsSphere3[1,:],
    mode='markers',
    marker=dict(size=3, color='green'),
    name='Sphere 3'
))

fig.update_layout(
    title='Projeizierte Datenpunkte, Berechnetes 2D Kamerabild ',
    xaxis_title='X (mm)',
    yaxis_title='Y (mm)',
    xaxis=dict(range=[-2000, 2000]),
    yaxis=dict(range=[-2000, 2000], scaleanchor="x", scaleratio=1),
    width=600,
    height=600
)
fig.show()

### Projektion eines Einzelnen Punktes

Nachfolgende Zeilen illustrieren, wie ein einzelner Punkt projiziert werden kann.

In [11]:
# Definition des Punktes in homogenen Koordinaten
point_world_hom = np.array([[20], [10], [50], [1]])

# Projektion des Punktes ins Kamerakoordinaten
point_image_hom = intCamMat@extCamMat @ point_world_hom

# Homogene Koordinaten in 2D Bildkoordinaten umwandeln
point_image = hom2Cart(point_image_hom)

print(f'Ausgangspunkt {point_world_hom}')
print(f'Bildpunkt in homogenen Koordinaten {point_image_hom}')
print(f'Bildpunkt karthesisch {point_image}')

Ausgangspunkt [[20]
 [10]
 [50]
 [ 1]]
Bildpunkt in homogenen Koordinaten [[  72000.        ]
 [-156831.94658174]
 [   1526.49793337]]
Bildpunkt karthesisch [[  47.16678511]
 [-102.73970449]]
